# Real vs Synthetic Face Classification

This notebook is the cleaned portfolio version of an academic machine-learning project. It compares two feature representations, **HOG** and **raw colour pixels**, with two classical classifiers, **KNN** and **linear SVM**.

The original project used a balanced dataset of 200 images: 100 real faces and 100 synthetic faces. A separate set of 10 images was used as a small external generalisation check.

> The image dataset is not included in this public repository. See `../data/README.md` for the expected folder structure.

## 1. Imports and paths

The original notebook used Google Drive-specific paths. This version uses relative paths so the project can run locally after the dataset is placed in the `data/` folder.

In [ ]:
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from skimage.feature import hog
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

IMG_SIZE = (128, 128)
CLASS_NAMES = ['Real', 'Synthetic']
LABEL_MAP = {0: 'Real', 1: 'Synthetic'}
VALID_EXTENSIONS = {'.jpg', '.jpeg', '.png'}

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_DIR = PROJECT_ROOT / 'data'
REAL_DIR = DATA_DIR / 'RealFace'
SYN_DIR = DATA_DIR / 'SynFace'
TEST_REAL_DIR = DATA_DIR / 'test_new' / 'RealFace'
TEST_SYN_DIR = DATA_DIR / 'test_new' / 'SynFace'

## 2. Image preprocessing and feature extraction

Each image is resized to **128 x 128**. HOG uses grayscale images and captures edge/texture patterns. Raw features retain colour information and flatten the resized image into one vector.

In [ ]:
def image_files(folder):
    folder = Path(folder)
    if not folder.exists():
        raise FileNotFoundError(
            f'Missing dataset folder: {folder}\n'
            'See data/README.md for the expected folder structure.'
        )
    return sorted(
        path for path in folder.iterdir()
        if path.is_file() and path.suffix.lower() in VALID_EXTENSIONS
    )


def process_image_hog(image_path, size=IMG_SIZE):
    image = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
    if image is None:
        return None
    image = cv2.resize(image, size) / 255.0
    return hog(
        image,
        pixels_per_cell=(8, 8),
        cells_per_block=(2, 2),
        visualize=False,
    )


def process_image_raw(image_path, size=IMG_SIZE):
    image = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
    if image is None:
        return None
    image = cv2.resize(image, size) / 255.0
    return image.flatten()

### Feature example from the original run

![Original, grayscale and HOG examples](../assets/feature_examples.png)

In [ ]:
def show_feature_examples():
    samples = {
        'Real': image_files(REAL_DIR)[0],
        'Synthetic': image_files(SYN_DIR)[0],
    }

    plt.figure(figsize=(10, 7))
    plot_position = 1

    for name, path in samples.items():
        colour = cv2.imread(str(path))
        colour = cv2.cvtColor(colour, cv2.COLOR_BGR2RGB)
        gray = cv2.resize(cv2.imread(str(path), cv2.IMREAD_GRAYSCALE), IMG_SIZE)

        _, hog_image = hog(
            gray / 255.0,
            pixels_per_cell=(8, 8),
            cells_per_block=(2, 2),
            visualize=True,
        )

        plt.subplot(2, 3, plot_position)
        plt.imshow(colour)
        plt.title(f'{name} - Original')
        plt.axis('off')

        plt.subplot(2, 3, plot_position + 1)
        plt.imshow(gray, cmap='gray')
        plt.title(f'{name} - Grayscale')
        plt.axis('off')

        plt.subplot(2, 3, plot_position + 2)
        plt.imshow(hog_image, cmap='gray')
        plt.title(f'{name} - HOG')
        plt.axis('off')

        plot_position += 3

    plt.tight_layout()
    plt.show()

# Uncomment after adding the dataset:
# show_feature_examples()

## 3. Load the training dataset

In [ ]:
def load_training_data():
    x_hog, x_raw, y = [], [], []
    folders = {0: REAL_DIR, 1: SYN_DIR}

    for label, folder in folders.items():
        for path in image_files(folder):
            hog_features = process_image_hog(path)
            raw_features = process_image_raw(path)

            if hog_features is not None and raw_features is not None:
                x_hog.append(hog_features)
                x_raw.append(raw_features)
                y.append(label)

    return np.asarray(x_hog), np.asarray(x_raw), np.asarray(y)

# Run after adding the dataset:
# X_hog, X_raw, y = load_training_data()
# print('HOG feature shape:', X_hog.shape)
# print('Raw feature shape:', X_raw.shape)
# print('Class balance -> Real:', np.sum(y == 0), 'Synthetic:', np.sum(y == 1))

## 4. Train and evaluate KNN and SVM

KNN tests odd values of `k` from 1 to 49 using 10-fold cross-validation on the training split. The final KNN and linear SVM models are then evaluated on the held-out test split.

The cleaned version uses `stratify=y` when creating the test split so the class balance is preserved.

In [ ]:
def run_pipeline(X, y, feature_name):
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
        stratify=y,
    )

    k_range = list(range(1, 50, 2))
    mean_scores = []

    for k in k_range:
        knn_cv = KNeighborsClassifier(n_neighbors=k)
        scores = cross_val_score(knn_cv, X_train, y_train, cv=10, scoring='accuracy')
        mean_scores.append(scores.mean())

    best_k = k_range[int(np.argmax(mean_scores))]
    print(f'{feature_name} - best K: {best_k}')

    plt.figure(figsize=(8, 5))
    plt.plot(k_range, mean_scores, marker='o')
    plt.xlabel('Number of Neighbors (K)')
    plt.ylabel('Mean Cross-Validation Accuracy')
    plt.title(f'{feature_name} - Cross-Validation Accuracy vs K')
    plt.xticks(k_range)
    plt.grid()
    plt.show()

    knn = KNeighborsClassifier(n_neighbors=best_k)
    knn.fit(X_train, y_train)
    knn_pred = knn.predict(X_test)

    svm = SVC(kernel='linear', class_weight='balanced')
    svm.fit(X_train, y_train)
    svm_pred = svm.predict(X_test)

    knn_acc = accuracy_score(y_test, knn_pred)
    svm_acc = accuracy_score(y_test, svm_pred)
    print(f'KNN test accuracy: {knn_acc * 100:.2f}%')
    print(f'SVM test accuracy: {svm_acc * 100:.2f}%')

    fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
    for ax, name, pred in zip(axes, ['KNN', 'SVM'], [knn_pred, svm_pred]):
        cm = confusion_matrix(y_test, pred)
        sns.heatmap(
            cm,
            annot=True,
            fmt='d',
            cmap='Blues',
            ax=ax,
            xticklabels=CLASS_NAMES,
            yticklabels=CLASS_NAMES,
        )
        ax.set_title(f'{feature_name} - {name}')
        ax.set_xlabel('Predicted')
        ax.set_ylabel('Actual')

    plt.tight_layout()
    plt.show()
    return knn, svm, knn_acc, svm_acc

# Example once X_hog, X_raw and y are loaded:
# hog_knn, hog_svm, hog_acc_knn, hog_acc_svm = run_pipeline(X_hog, y, 'HOG features')
# raw_knn, raw_svm, raw_acc_knn, raw_acc_svm = run_pipeline(X_raw, y, 'Raw colour pixels')

## 5. Recorded results from the original project

These are the recorded results from the submitted run. A fresh run of the cleaned notebook can differ slightly because the cleaned split preserves class balance with stratification.

| Feature representation | KNN test accuracy | SVM test accuracy |
|---|---:|---:|
| HOG | 80.0% | **92.5%** |
| Raw colour pixels | 82.5% | 90.0% |

Five-fold cross-validation:

| Model | Mean accuracy | Std. dev. |
|---|---:|---:|
| HOG-KNN | 77.5% | 4.18% |
| **HOG-SVM** | **93.5%** | 3.74% |
| Raw-KNN | 84.5% | **3.32%** |
| Raw-SVM | 88.5% | 3.39% |

### Original confusion matrices

**HOG**

![HOG confusion matrices](../assets/hog_confusion_matrices.png)

**Raw colour pixels**

![Raw confusion matrices](../assets/raw_confusion_matrices.png)

## 6. Five-fold model comparison

In [ ]:
def compare_cross_validation(models, y):
    cv_results = {}

    for name, (model, X) in models.items():
        scores = cross_val_score(model, X, y, cv=5, scoring='accuracy')
        cv_results[name] = scores
        print(f'{name}: {scores}')
        print(f'Mean = {scores.mean() * 100:.2f}% Std = {scores.std() * 100:.2f}%\n')

    best_mean = max(cv_results, key=lambda name: cv_results[name].mean())
    best_std = min(cv_results, key=lambda name: cv_results[name].std())

    print(f'Highest mean accuracy: {best_mean} ({cv_results[best_mean].mean() * 100:.2f}%)')
    print(f'Most consistent: {best_std} (std {cv_results[best_std].std() * 100:.2f}%)')
    return cv_results

# Example after training:
# models = {
#     'HOG-KNN': (hog_knn, X_hog),
#     'HOG-SVM': (hog_svm, X_hog),
#     'RAW-KNN': (raw_knn, X_raw),
#     'RAW-SVM': (raw_svm, X_raw),
# }
# cv_results = compare_cross_validation(models, y)

## 7. External-image evaluation

The original project also checked 10 unseen images, five real and five synthetic. For privacy, this public portfolio version reports **aggregate accuracy only**. Individual external test images, filenames, and per-image predictions are intentionally omitted.

Because the sample contains only 10 images, it should be treated as a small generalisation check rather than a reliable estimate of real-world performance.


In [ ]:
def evaluate_external_images(hog_knn, hog_svm, raw_knn, raw_svm):
    """Return aggregate accuracy only; do not expose individual test samples."""
    test_folders = {0: TEST_REAL_DIR, 1: TEST_SYN_DIR}
    correct = {'HOG-KNN': 0, 'HOG-SVM': 0, 'RAW-KNN': 0, 'RAW-SVM': 0}
    total = 0

    for true_label, folder in test_folders.items():
        for path in image_files(folder):
            hog_features = process_image_hog(path)
            raw_features = process_image_raw(path)
            if hog_features is None or raw_features is None:
                continue

            hog_features = hog_features.reshape(1, -1)
            raw_features = raw_features.reshape(1, -1)

            predictions = {
                'HOG-KNN': hog_knn.predict(hog_features)[0],
                'HOG-SVM': hog_svm.predict(hog_features)[0],
                'RAW-KNN': raw_knn.predict(raw_features)[0],
                'RAW-SVM': raw_svm.predict(raw_features)[0],
            }

            total += 1
            for name, prediction in predictions.items():
                correct[name] += int(prediction == true_label)

    accuracies = {}
    print(f'External-image accuracy on {total} images:')
    for name, count in correct.items():
        accuracy = count / total if total else 0.0
        accuracies[name] = accuracy
        print(f'{name}: {count}/{total} = {accuracy * 100:.2f}%')

    return accuracies

# Example after training:
# external_accuracy = evaluate_external_images(hog_knn, hog_svm, raw_knn, raw_svm)


### Recorded external-image result

Only aggregate accuracy is shown in the public portfolio version.

| Model | Correct | Accuracy |
|---|---:|---:|
| HOG-KNN | 4 / 10 | 40% |
| HOG-SVM | 7 / 10 | 70% |
| **Raw-KNN** | **8 / 10** | **80%** |
| Raw-SVM | 7 / 10 | 70% |

> Individual external test images and per-image predictions are intentionally excluded for privacy.


## 8. Limitations and next steps

This project is a classical-ML baseline rather than a production deepfake detector. The main limitations are the small dataset, the 10-image external test set, and the possibility that backgrounds, lighting, clothing, accessories, and colour style influence the predictions.

Useful next steps would be:

- larger and more varied training and test datasets
- face detection and alignment before feature extraction
- image augmentation
- combining HOG with colour histograms
- broader model/hyperparameter tuning
- comparison with a CNN or transfer-learning baseline

The most important lesson from the project is that **high held-out accuracy does not automatically mean strong performance on unfamiliar image styles**.